# 1 · Eavesdropper on FABRIC — QBER vs Eve's tap fraction over the real WAN

The FABRIC counterpart to `10_eavesdropper`. Instead of an in-process sweep, this runs the
real two-process BB84 (`node_runner`) across the **alice and bob slice nodes**, with the
intercept-resend Eve (`--eve-fraction`) on the channel, and measures QBER on real hardware.

Expect the same physics — QBER ≈ 0.25·f plus the channel's intrinsic ~(1−F)/2 — now with
real WAN timing in the loop. Prereq: run notebook fabric/01 (slice + switch) first.

## 1 · Configuration

In [ ]:
SLICE_NAME = 'qfabric-bb84-2'                       # same slice as notebooks fabric/01 / sequence/01
SCENARIO   = 'validation/scenarios/fabric_1km.yml'
BMV2_IMAGE = 'ghcr.io/kthare10/qfabric-bmv2:latest'

# Channel mode — matches notebook sequence/01. Use 'raw'+'switch' (the P4 path) if your slice
# already runs it; 'tcp' is a simpler switch-free option (descriptors over the link).
TRANSPORT = 'raw'      # 'raw' (0x7101 through BMv2) | 'tcp' (no switch)
LOSS      = 'switch'   # 'switch' | 'model' | 'none' | 'auto'
NUM_PULSES      = 20000
SAMPLE_FRACTION = 0.2

## 2 · Load the slice

In [ ]:
import os, sys, json
from pathlib import Path
import matplotlib.pyplot as plt

PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'qne').is_dir())
sys.path.insert(0, str(PROJECT_DIR)); sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
import deploy_fabric as df
from qne.config import ScenarioConfig

cfg = ScenarioConfig.from_yaml(PROJECT_DIR / SCENARIO)
fablib = df.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
slice_obj.show()

## 3 · Ship code + build runtime + arm the switch

In [ ]:
df.upload_project(slice_obj)
if BMV2_IMAGE:
    os.environ['QFABRIC_BMV2_IMAGE'] = BMV2_IMAGE
df.setup_sequence_runtime(slice_obj)

USE_SWITCH = (TRANSPORT == 'raw' and LOSS in ('switch', 'auto'))
if USE_SWITCH:
    df.configure_switch(slice_obj, cfg.loss_threshold_u32)
    print('switch armed')
else:
    print(f'no switch needed (transport={TRANSPORT}, loss={LOSS})')

## 4 · Sweep Eve's tap fraction on the slice

In [ ]:
EVE_FRACTIONS = [0.0, 0.25, 0.5, 0.75, 1.0]
rows = []
for f in EVE_FRACTIONS:
    print(f"\n### eve_fraction = {f}")
    a, b = df.run_sequence_bb84(
        slice_obj, transport=TRANSPORT, loss=LOSS,
        num_pulses=NUM_PULSES, fidelity=cfg.channel.polarization_fidelity,
        efficiency=cfg.detector.efficiency, dark_count_rate=cfg.detector.dark_count_rate,
        distance_km=cfg.channel.distance_km, attenuation=cfg.channel.attenuation_db_per_km,
        sample_fraction=SAMPLE_FRACTION, eve_fraction=f, reconcile=False)
    if a:
        rows.append({'f': f, 'qber': a['qber'], 'secure_fraction': a['secure_fraction'],
                     'sifted': a['sifted_bits']})

## 5 · Plot (measured on FABRIC)

In [ ]:
F0 = cfg.channel.polarization_fidelity
q0 = (1 - F0) / 2
fs = [r['f'] for r in rows]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(fs, [r['qber'] for r in rows], 'o-', label='measured (FABRIC)')
ax1.plot(fs, [q0 + 0.25*f for f in fs], 'k--', lw=1, label=f'theory q0+0.25f (q0={q0:.3f})')
ax1.axhline(0.11, color='crimson', ls=':', label='~11% threshold')
ax1.set(xlabel="Eve's tap fraction f", ylabel='QBER', title='Eavesdropping on the real WAN')
ax1.legend(fontsize=8); ax1.grid(alpha=.3)
ax2.plot(fs, [r['secure_fraction'] for r in rows], 'o-', color='#F06FA6')
ax2.set(xlabel="Eve's tap fraction f", ylabel='secure fraction', title='Secure key rate collapses')
ax2.grid(alpha=.3)
fig.tight_layout()
out = PROJECT_DIR / 'paper' / 'figures' / 'eve_fabric.png'
out.parent.mkdir(parents=True, exist_ok=True); fig.savefig(out, dpi=150, bbox_inches='tight')
print('saved', out); plt.show()

## 6 · Verify

In [ ]:
checks = []
def rec(n, ok, d=''):
    checks.append(ok); print(f"  [{'PASS' if ok else 'FAIL'}] {n}" + (f' — {d}' if d else ''))
byf = {r['f']: r for r in rows}
if 0.0 in byf: rec('no Eve: low QBER', byf[0.0]['qber'] < 0.05, f"QBER={byf[0.0]['qber']:.4f}")
if 1.0 in byf: rec('full intercept: QBER ~0.25', abs(byf[1.0]['qber'] - 0.25) < 0.05, f"QBER={byf[1.0]['qber']:.4f}")
rec('QBER rises with f', all(rows[i]['qber'] <= rows[i+1]['qber'] + 0.03 for i in range(len(rows)-1)))
print('\nALL CHECKS PASSED' if checks and all(checks) else '\nSOME CHECKS FAILED — see /tmp/seq_*.log on the nodes')

## Notes

- Real-hardware QBER has more run-to-run scatter than the local sim — widen tolerances or average a few runs per point if needed.
- This is the intern stretch goal made concrete; compare the curve to the local `10_eavesdropper` sweep to see how (little) the WAN changes the *physics*.